In [1]:
import numpy as np
import pandas as pd
import json
from pathlib import Path
from sklearn.model_selection import StratifiedKFold
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

PROJECT_ROOT    = Path(r"A:\Coding\PycharmProjects\cryptoguard")
BLOCKCHAIN_PATH = PROJECT_ROOT / "data" / "blockchain" / "cleaned.csv"
RESULTS_DIR     = PROJECT_ROOT / "outputs" / "results"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

df     = pd.read_csv(BLOCKCHAIN_PATH)
df['label'] = df['label'].astype(int)
texts  = df['text'].tolist()
labels = df['label'].values

skf      = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
results  = []

for fold, (train_idx, val_idx) in enumerate(skf.split(texts, labels), 1):
    train_texts  = [texts[i] for i in train_idx]
    val_texts    = [texts[i] for i in val_idx]
    train_labels = labels[train_idx]
    val_labels   = labels[val_idx]

    vec     = TfidfVectorizer(max_features=50000, ngram_range=(1, 2), sublinear_tf=True)
    X_train = vec.fit_transform(train_texts)
    X_val   = vec.transform(val_texts)
    clf     = LogisticRegression(C=1.0, max_iter=1000, random_state=42)
    clf.fit(X_train, train_labels)

    y_pred = clf.predict(X_val)
    y_prob = clf.predict_proba(X_val)[:, 1]

    metrics = {
        "fold":      fold,
        "accuracy":  round(accuracy_score(val_labels,  y_pred), 4),
        "precision": round(precision_score(val_labels, y_pred, zero_division=0), 4),
        "recall":    round(recall_score(val_labels,    y_pred, zero_division=0), 4),
        "f1":        round(f1_score(val_labels,        y_pred, zero_division=0), 4),
        "roc_auc":   round(roc_auc_score(val_labels,   y_prob), 4),
    }
    results.append(metrics)
    print(f"Fold {fold} — F1={metrics['f1']:.4f}  Acc={metrics['accuracy']:.4f}  AUC={metrics['roc_auc']:.4f}")

print(f"\nTF-IDF 5-Fold CV:")
for metric in ["accuracy", "precision", "recall", "f1", "roc_auc"]:
    vals = [r[metric] for r in results]
    print(f"  {metric:10s}: {np.mean(vals):.4f} +/- {np.std(vals):.4f}")

with open(RESULTS_DIR / "cv_tfidf.json", "w") as f:
    json.dump(results, f, indent=2)

Fold 1 — F1=0.9832  Acc=0.9830  AUC=0.9986
Fold 2 — F1=0.9875  Acc=0.9874  AUC=0.9985
Fold 3 — F1=0.9837  Acc=0.9836  AUC=0.9985
Fold 4 — F1=0.9862  Acc=0.9862  AUC=0.9993
Fold 5 — F1=0.9813  Acc=0.9811  AUC=0.9981

TF-IDF 5-Fold CV:
  accuracy  : 0.9843 +/- 0.0023
  precision : 0.9791 +/- 0.0056
  recall    : 0.9897 +/- 0.0022
  f1        : 0.9844 +/- 0.0022
  roc_auc   : 0.9986 +/- 0.0004
